# 📊 Module 10: Model Evaluation & Benchmarking
**Movie Recommendation System Using Machine Learning**

---

### 🎯 Objectives
1. Understand why recommender system evaluation requires both **Rating Accuracy** and **Ranking Quality** metrics.
2. Calculate and explain **RMSE** (Root Mean Squared Error) and **MAE** (Mean Absolute Error).
3. Calculate and explain **Precision@K** and **Recall@K** using a relevance threshold ($r \ge 3.5$).
4. Calculate **Catalog Coverage** to measure recommendation diversity.
5. Benchmark Item-Based CF against User-Based CF.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")
from src.recommender import MovieRecommender
from src.evaluation import run_full_evaluation

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("Environment ready!")


## 1. Mathematical Formulas for Metrics

### 1. Root Mean Squared Error (RMSE)
$$\text{RMSE} = \sqrt{\frac{1}{|T|} \sum_{(u,i) \in T} (r_{u,i} - \hat{r}_{u,i})^2}$$
Penalizes large errors heavily.

### 2. Mean Absolute Error (MAE)
$$\text{MAE} = \frac{1}{|T|} \sum_{(u,i) \in T} |r_{u,i} - \hat{r}_{u,i}|$$
Linear, interpretable average star error.

### 3. Precision@K and Recall@K
- Item is *Relevant* if actual rating $\ge 3.5$.
$$\text{Precision@K} = \frac{|\text{Recommended}_K \cap \text{Relevant}|}{K}$$
$$\text{Recall@K} = \frac{|\text{Recommended}_K \cap \text{Relevant}|}{|\text{Relevant}|}$$

### 4. Catalog Coverage
$$\text{Coverage} = \frac{|\bigcup_u \text{Recommended}_u|}{|\text{Catalog Items}|} \times 100\%$$


In [ ]:
# Load model and test ratings
model = MovieRecommender.load("../models/recommender_model.joblib")
test_ratings = pd.read_csv("../dataset/test_ratings.csv")

print(f"Loaded {len(test_ratings):,} held-out test ratings.")


In [ ]:
# Run complete benchmark
results = run_full_evaluation(
    recommender=model,
    test_ratings_df=test_ratings,
    k=10,
    relevance_threshold=3.5,
    num_eval_users=80,
    max_rating_pairs=2000
)

results_df = pd.DataFrame(results).T
display(results_df)


## 2. Visualization of Evaluation Benchmark

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# RMSE & MAE
models = ["Item-Based CF", "User-Based CF"]
rmse = [results["item_based"]["RMSE"], results["user_based"]["RMSE"]]
mae = [results["item_based"]["MAE"], results["user_based"]["MAE"]]

x = np.arange(len(models))
w = 0.35
ax1.bar(x - w/2, rmse, w, label="RMSE", color="#4C72B0")
ax1.bar(x + w/2, mae, w, label="MAE", color="#55A868")
ax1.set_title("Rating Prediction Error (Lower is Better)")
ax1.set_ylabel("Error (Stars)")
ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.legend()

# Coverage
cov = [results["item_based"]["Coverage@10 (%)"], results["user_based"]["Coverage@10 (%)"]]
ax2.bar(models, cov, color=["#C44E52", "#8172B2"])
ax2.set_title("Catalog Coverage @ 10 (Higher is Better)")
ax2.set_ylabel("Catalog Coverage (%)")

plt.tight_layout()
plt.show()


## 3. Summary & Takeaways
- **Item-Based CF** achieves significantly lower prediction error (lower RMSE and MAE) because item similarity patterns are dense and robust across users.
- **User-Based CF** exhibits higher catalog coverage because users possess diverse idiosyncratic tastes that surface a wider variety of recommendations.
